# 05 · What does the EMG encode, and at what lag?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 4 · Link to movement: what does EMG encode (angle, velocity, change), at what lag?**

**Read first:** [Hadidi et al. 2026](https://arxiv.org/abs/2603.08212) (position vs velocity decoding on this dataset; the 15.63° / 18.87° numbers), Salter et al. 2024, *emg2pose* ([arXiv:2412.02725](https://arxiv.org/abs/2412.02725)), and [`EVALUATION.md`](../EVALUATION.md) for why an earlier version of this analysis was uninformative.

**We expect:** the EMG envelope **at one instant** says little about the pose, but its **recent history** (100–300 ms) does; a decoder using that history beats "always guess the median pose" and a time-shifted null on held-out stages. **Wrong if:** lagged EMG is no better than the shifted null, or no better than each recording's own median pose.

**Where this sits.** Stage 4 asks what the EMG tells us about the hand. Notebook 06 (stage 4b) then asks *which part* of the EMG carries it (the shared EMG–pose subspace), notebook 07 (stage 5) models the dynamics, and notebook 08 (stage 6) tests the bottlenecks with the decoder built here.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref05.py`](ref05.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** 🧪 **H2** of the research plan (EMG explains movement) and the decoder that the Aim 2 pilot (notebook 08) needs, with linear decoders on one person.

**Goal:** find out what the EMG envelope can tell us about the pose (angle or velocity, now or with history), measured honestly against two baselines and a null.

**Why:** an earlier version compared bottlenecks using a decoder that was no better than guessing the median pose, so every condition looked the same. A bottleneck can only be judged once the decoder carries information.

**Milestones**
1. Baselines: global median **and** each recording's own median, scored per block
2. Ridge from the current EMG, then with EMG history (lag sweep, log-EMG)
3. Angle vs velocity; decoding (EMG → pose) vs encoding (pose → EMG)
4. Held-out stages (leave-one-stage-out) and a time-shifted null, with confidence intervals over blocks

**Gate (decides the next stage):** stage 4 passes if the lagged decoder beats the global median **and** the shifted null on held-out stages (95% CI of the paired difference below 0), and we know whether it beats each recording's own median.

**Done when:** the stage 4 gate table filled in, with a pass / fail and a reason.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "05_link_to_movement"
assert (_d / "ref05.py").exists(), f"{_d}/ref05.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref05", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref05 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees (low-passed at 6 Hz, data version 2)
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed (dropouts padded by 40 ms)
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec, filename = d["stage"], d["side"], d["filename"]   # (30,) per recording

train, test = split == 0, split == 1
tr = train & pose_valid    # (N,) bool: training frames with a usable pose
te = test & pose_valid     # (N,) bool: test frames with a usable pose
block = protocol.session_ids(stage_of_rec, rec, filename)   # (N,) independent block (0–14) of every frame
N_BLOCKS = 15
SEED = 0
results: dict[str, dict[str, float]] = {}   # e.g. results["global median"] = {"angle_deg": …, "r2": …}
print(f"{len(emg)} frames | {tr.sum()} train / {te.sum()} test frames with valid pose | {N_BLOCKS} blocks")

### The variables you start with
`N` = 105,484 frames (about 35 minutes at 50 frames/s). Every row is one frame, 20 ms long. Boolean arrays are
**masks**: `emg[tr]` keeps only those rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float64 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float64 | joint angles in **degrees** (low-passed at 6 Hz in notebook 01, Task 5b) |
| `pose_valid` | (N,) | bool | False where pose tracking failed, including 40 ms either side of each dropout |
| `rec` | (N,) | int16 | recording (0–29) of each frame |
| `tr`, `te` | (N,) | bool | **time split**: the first 80% / last 20% of every recording, valid pose only |
| `block` | (N,) | int | the **independent block** (0–14) of each frame: the left- and right-hand recordings of a block were made **at the same time**, so they count as one unit |
| `filename` | (30,) | str | file name per recording; `protocol` uses it to find the blocks |
| `results` | dict | | name → `{"angle_deg": …, "r2": …}`, filled as you go |

**Why blocks?** This is one person, and the two hands of a block were recorded simultaneously, so there are only
**15 independent units**. Frames 20 ms apart are nearly copies of each other. Every confidence interval in this
notebook is therefore computed **over blocks** (`protocol.session_bootstrap_ci`), never over frames.

Letters in the docstrings: **T** = frames, **d** = input features, **J** = output columns (20 angles).

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
# Stage 4: what does the EMG encode?
Three ideas run through this part:

1. **Two baselines, not one.** "Always guess the median pose" is the obvious one. But each recording is one task
   with its own typical hand shape, so **each recording's own median pose** is a much stronger baseline on the time
   split. If a decoder can't beat it, it may only be recognising *which task* is going on, not tracking the hand.
2. **A null that keeps everything except the alignment.** `protocol.circular_shift_null` slides the EMG of each
   recording in time by a random amount (at least 5 s), wrapping around at the end. The EMG keeps all its own
   structure, but no longer lines up with the pose. A real decoder must beat a decoder trained on this.
3. **Held-out stages.** The time split tests the *same* tasks later on. `protocol.leave_one_stage_out(...,
   filename=filename)` holds out a whole task (and anything recorded at the same time) and trains on the rest:
   can the decoder handle a movement it has never seen?

---
## Task 1 · Two ways to score a pose prediction
**Angular error (°)** is the mean absolute difference between true and predicted joint angles, averaged over
every joint and frame. It is the unit published papers use (the best model on the full dataset: 15.63°).

**R²** asks: how much better is the prediction than just guessing each joint's average? For one joint `j`:

$$R^2_j = 1 - \frac{\sum_t (y_{tj} - \hat y_{tj})^2}{\sum_t (y_{tj} - \bar y_j)^2}$$

- **1** = perfect. **0** = no better than always predicting that joint's test-set mean. **Negative** = worse than that.
- Compute it **per joint** (sums run down the time axis), then **average the 20 values**. That's what
  scikit-learn's `r2_score` does by default, so joints that hardly move don't get drowned out by big ones.

Angular error is in degrees you can picture, but a small error is easy if the hand barely moves. R² says whether
you captured the *movement*, not just the average pose.

In [ ]:
def angular_error(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """Mean absolute joint-angle error.

    Parameters
    ----------
    Y     : (T, 20) float, true joint angles in degrees
    Y_hat : (T, 20) float, predicted joint angles in degrees

    Returns
    -------
    float, degrees, averaged over all joints and frames. Check: angular_error(Y, Y) → 0.0
    """
    # TODO
    raise NotImplementedError


def r2(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """R², computed per column then averaged over columns.

    Parameters
    ----------
    Y, Y_hat : (T, J) float

    Returns
    -------
    float, at most 1. Per column j: 1 - sum_t (Y - Y_hat)^2 / sum_t (Y - mean_t Y)^2, then the mean over j.
    """
    # TODO
    raise NotImplementedError


def score(Y: np.ndarray, Y_hat: np.ndarray) -> dict[str, float]:
    """Both metrics at once: {"angle_deg": angular_error(...), "r2": r2(...)}."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `angular_error`: absolute difference at every entry, then the mean of all of them (one number).
2. `r2`: two sums **down the time axis** (`axis=0`), giving one value per joint: the squared error, and the
   squared distance of `Y` from its own column mean. Then `1 - ratio` per joint, then the mean of those.
3. `score` just puts the two into a dictionary.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.mean(a)` with no axis → one number. `np.sum(a, axis=0)` → one number per column.
- `Y.mean(axis=0)` has shape (J,); `Y - Y.mean(axis=0)` subtracts each column's mean from that column.

```python
Y = np.array([[1.0, 10.0], [3.0, 10.0], [5.0, 13.0]])
np.sum((Y - Y.mean(axis=0)) ** 2, axis=0)    # array([8., 6.]): one total per column
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def angular_error(Y, Y_hat):
    return float(np.mean(np.abs(____ - ____)))

def r2(Y, Y_hat):
    ss_res = np.sum((Y - Y_hat) ** 2, axis=____)
    ss_tot = np.sum((Y - Y.mean(axis=____)) ** 2, axis=____)
    return float(np.mean(1 - ____ / ____))

def score(Y, Y_hat):
    return {"angle_deg": angular_error(Y, Y_hat), "r2": ____}
```

</details>

In [ ]:
# ✅ Check Task 1
from sklearn.metrics import r2_score     # only to confirm your r2 matches the library's
rng_chk = np.random.default_rng(0)
Y_toy = rng_chk.normal(size=(40, 20)) * 10
Y_hat_toy = Y_toy + rng_chk.normal(size=(40, 20)) * 3
assert angular_error(Y_toy, Y_toy) == 0.0
assert abs(angular_error(Y_toy, Y_toy + 2) - 2.0) < 1e-9, "an error of 2° everywhere should give 2.0"
assert abs(r2(Y_toy, Y_toy) - 1.0) < 1e-12, "a perfect prediction should give R² = 1"
assert abs(r2(Y_toy, np.tile(Y_toy.mean(axis=0), (40, 1)))) < 1e-12, "predicting each column's mean should give R² = 0"
assert abs(r2(Y_toy, Y_hat_toy) - ref.r2(Y_toy, Y_hat_toy)) < 1e-9, \
    f"r2 = {r2(Y_toy, Y_hat_toy):.4f}, expected {ref.r2(Y_toy, Y_hat_toy):.4f}: per column, then the mean?"
assert abs(r2(Y_toy, Y_hat_toy) - r2_score(Y_toy, Y_hat_toy)) < 1e-9
assert score(Y_toy, Y_hat_toy) == ref.score(Y_toy, Y_hat_toy)
print(f"✅ Metrics work (and your R² matches scikit-learn's r2_score: {r2(Y_toy, Y_hat_toy):.3f})")

✍️ **Q1.** A decoder gets 12° angular error but R² = −0.1. Another gets 16° but R² = 0.3. Describe a hand recording where
each could happen. Which number would you report to show a decoder *tracks movement*?

> *Your answer:*

---
## Task 2 · Two baselines, scored per block
Before any decoder, ask what you get **without EMG at all**.

- **Global median:** one "typical" pose (the median of all training poses), predicted for every test frame.
  Why the **median**? Angular error is an *absolute* error, and the median is the single value that makes total
  absolute error smallest.
- **Per-recording median:** each recording's own median **training** pose, predicted for that recording's test
  frames. This uses no EMG, only "which recording is this?". It is a fair baseline for the **time split** (the
  test frames come from the same recording), but not for held-out stages (a new task has no training frames).

Then a helper you'll use all notebook: `per_block_error` turns one prediction into **15 numbers, one per block**,
so you can put a confidence interval on it, and compare two methods **block by block** (a *paired* comparison).

In [ ]:
median_pose = None   # (20,) float: the median of pose[tr] over frames, one value per joint
Y_hat_base  = None   # (N, 20): median_pose repeated for EVERY frame (select rows with a mask later)
# TODO: fill both, then results["global median"] = score(pose[te], Y_hat_base[te])


def per_recording_median(Y: np.ndarray, rec: np.ndarray, fit_mask: np.ndarray) -> np.ndarray:
    """Each recording's own median, as a prediction for all of its frames.

    Parameters
    ----------
    Y        : (N, J) float, e.g. pose
    rec      : (N,) int, recording of each frame
    fit_mask : (N,) bool, frames allowed to set the medians (e.g. tr)

    Returns
    -------
    (N, J) float: row t = the median of Y over fit_mask frames of rec[t]'s recording
    (the global median of Y[fit_mask] for a recording with no fit_mask frames).
    """
    # TODO
    raise NotImplementedError


def per_block_error(Y: np.ndarray, Y_hat: np.ndarray, mask: np.ndarray, block: np.ndarray,
                    n_blocks: int = 15) -> np.ndarray:
    """(n_blocks,) angular error over the mask frames of each block; NaN for a block with no such frames."""
    # TODO
    raise NotImplementedError


Y_hat_rec = None     # (N, 20): TODO per_recording_median(pose, rec, tr)
# TODO: results["per-recording median"] = score(pose[te], Y_hat_rec[te])

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Global: the median of `pose[tr]` down the time axis, then `np.tile` it into N rows.
2. `per_recording_median`: start from the global prediction; loop over recordings; where a recording has
   `fit_mask` frames, overwrite its rows with the median of those frames.
3. `per_block_error`: loop over blocks `b`; the frames are `mask & (block == b)`; use your `angular_error`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.median(a, axis=0)` → one median per column; `np.tile(row, (n, 1))` → n copies stacked as rows.
- `np.unique(rec)` → each recording id once. `np.full(15, np.nan)` → an array of NaN to fill in.

```python
rec_toy = np.array([0, 0, 1, 1]); Y_toy2 = np.array([[1.0], [3.0], [10.0], [20.0]])
np.median(Y_toy2[rec_toy == 1], axis=0)     # array([15.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
median_pose = np.median(pose[____], axis=____)
Y_hat_base = np.tile(median_pose, (len(pose), ____))
results["global median"] = score(pose[te], Y_hat_base[____])

def per_recording_median(Y, rec, fit_mask):
    out = np.tile(np.median(Y[fit_mask], axis=0), (len(Y), 1))
    for r in np.unique(rec):
        m = (rec == r) & ____
        if m.any():
            out[rec == r] = np.median(Y[____], axis=0)
    return out

def per_block_error(Y, Y_hat, mask, block, n_blocks=15):
    out = np.full(n_blocks, np.nan)
    for b in range(n_blocks):
        m = mask & (block == ____)
        if m.any():
            out[b] = angular_error(Y[m], Y_hat[____])
    return out

Y_hat_rec = per_recording_median(pose, rec, ____)
results["per-recording median"] = score(pose[te], Y_hat_rec[te])
```

</details>

In [ ]:
# ✅ Check Task 2
assert median_pose.shape == (20,) and Y_hat_base.shape == pose.shape
assert np.allclose(median_pose, np.median(pose[tr], axis=0)), "use the TRAIN frames with valid pose (tr), axis=0"
assert np.allclose(Y_hat_rec, ref.per_recording_median(pose, rec, tr)), "per-recording medians differ from the reference"
assert np.allclose(Y_hat_rec, protocol.per_recording_median_baseline(pose, rec, tr)), "…and from the shared protocol"
e_glob = per_block_error(pose, Y_hat_base, te, block)
assert e_glob.shape == (N_BLOCKS,)
assert np.allclose(e_glob, ref.per_block_error(pose, Y_hat_base, te, block), equal_nan=True)
e_rec = per_block_error(pose, Y_hat_rec, te, block)
for name in ["global median", "per-recording median"]:
    print(f"{name:22s} {results[name]['angle_deg']:6.2f}°   R² = {results[name]['r2']:6.3f}")
m, lo, hi = protocol.session_bootstrap_ci(e_rec - e_glob)
print(f"✅ per-recording minus global, per block: {m:+.2f}° (95% CI {lo:+.2f} to {hi:+.2f})")

### 📊 The two baselines, block by block
One pair of bars per block. **What to look for:** how much of the "error" disappears just by knowing which task
(recording) you are in. That gap is information a decoder could pick up from the EMG *without* tracking the hand.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
x = np.arange(N_BLOCKS)
ax.bar(x - 0.2, e_glob, 0.4, label="global median")
ax.bar(x + 0.2, e_rec, 0.4, label="per-recording median")
ax.set_xticks(x); ax.set_xlabel("block"); ax.set_ylabel("test angular error (°)"); ax.legend()
plt.tight_layout(); plt.show()

✍️ **Q2.** The per-recording median needs no EMG at all, yet it is a few degrees better than the global median. What does it
know that the global median doesn't? Why is it a fair baseline on the time split but **not** on held-out stages?

> *Your answer:*

---
## Task 3 · Ridge regression, written from scratch
A **linear decoder** predicts every joint angle as a weighted sum of the features plus an offset:
`Ŷ = F w + b`, with `F` (T × d) the features, `w` (d × J) the weights and `b` (J,) one offset per joint.

**Least squares** picks `w` and `b` to make the squared error `Σ (Y − F w − b)²` as small as possible.
**Ridge** adds a penalty on large weights:

$$\min_{w,b}\; \sum (Y - F w - b)^2 \;+\; \alpha \sum w^2$$

**Why the penalty?** EMG channels are strongly correlated (neighbouring electrodes hear the same muscles). Then
plain least squares can make huge weights that cancel each other out: +500 on one channel, −499 on its
neighbour. That fits the training noise and falls apart on new data. The `α Σ w²` term makes big weights
expensive, so the fit spreads the weight sensibly. `α = 0` is plain least squares; bigger `α` = more caution.

**Solving it.** Two steps:

1. **Centre** `F` and `Y`: subtract each column's mean, giving `Fc` and `Yc`. The offset `b` is only there to
   absorb averages; once everything has mean 0 there is nothing left for it to do, so it drops out of the
   problem. This is also how we keep `b` **out of the penalty**: shifting all the angles by 10° should move
   `b`, not cost anything.
2. Set the derivative of the cost with respect to `w` to 0 and rearrange (the *normal equations*):

$$(F_c^\top F_c + \alpha I)\, w = F_c^\top Y_c \qquad\Rightarrow\qquad b = \bar Y - \bar F\, w$$

Solve the first with `np.linalg.solve(A, B)`, which finds `w` with `A @ w = B`. Never compute an explicit
inverse (`np.linalg.inv`): `solve` is faster and more accurate. The `+ α I` also guarantees `A` is invertible,
even when channels are perfectly correlated.

**One `w` for all 20 joints at once:** `Fc.T @ Yc` is (d × J), so `solve` returns all J columns of `w` together.

In [ ]:
def ridge_fit(F: np.ndarray, Y: np.ndarray, alpha: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Ridge regression in closed form, with an unpenalised offset.

    Parameters
    ----------
    F     : (T, d) float, input features
    Y     : (T, J) float, targets
    alpha : float ≥ 0, the penalty on the size of the weights

    Returns
    -------
    w : (d, J) float, weights:  solve (Fcᵀ Fc + alpha I) w = Fcᵀ Yc  with Fc, Yc centred
    b : (J,) float, offset:     mean(Y) - mean(F) @ w
    """
    # TODO
    raise NotImplementedError


def ridge_predict(F: np.ndarray, w: np.ndarray, b: np.ndarray) -> np.ndarray:
    """(T, d) features → (T, J) predictions: F @ w + b."""
    # TODO
    raise NotImplementedError


def fit_and_score_ridge(F_train: np.ndarray, Y_train: np.ndarray, F_test: np.ndarray,
                        Y_test: np.ndarray, alpha: float = 1.0) -> dict[str, float]:
    """Fit ridge on the train rows, predict the test rows, return score(Y_test, prediction)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `ridge_fit`: column means of `F` and `Y`; centred copies; build `A = Fcᵀ Fc + α I` (d × d) and
   `B = Fcᵀ Yc` (d × J); `w = solve(A, B)`; then `b` from the means.
2. `ridge_predict`: one line, `F @ w + b` (`b` is added to every row by broadcasting).
3. `fit_and_score_ridge`: fit on train, predict test, `score` it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.eye(d)` → the d × d identity matrix `I`.
- `np.linalg.solve(A, B)` → the `X` with `A @ X = B`; `B` may have many columns.
- `f_mean @ w`: (d,) times (d, J) → (J,).

```python
A = np.array([[2.0, 0.0], [0.0, 4.0]]); B = np.array([[2.0], [8.0]])
np.linalg.solve(A, B)       # [[1.], [2.]]  because A @ [[1], [2]] = [[2], [8]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def ridge_fit(F, Y, alpha=1.0):
    f_mean, y_mean = F.mean(axis=0), Y.mean(axis=0)
    Fc, Yc = F - ____, Y - ____
    w = np.linalg.solve(Fc.T @ Fc + alpha * np.eye(F.shape[____]), Fc.T @ ____)
    b = y_mean - ____ @ w
    return w, b

def ridge_predict(F, w, b):
    return ____ @ ____ + ____

def fit_and_score_ridge(F_train, Y_train, F_test, Y_test, alpha=1.0):
    w, b = ridge_fit(F_train, Y_train, alpha)
    return score(____, ridge_predict(____, w, b))
```

</details>

In [ ]:
# ✅ Check Task 3
from sklearn.linear_model import Ridge    # only to confirm your ridge matches the library's
F_toy = rng_chk.random((200, 16)); Y_lin = F_toy @ rng_chk.normal(size=(16, 20)) + 5.0
w, b = ridge_fit(F_toy, Y_lin, alpha=0.0)
assert w.shape == (16, 20) and b.shape == (20,), f"shapes {w.shape}, {b.shape}; expected (16, 20), (20,)"
assert np.allclose(ridge_predict(F_toy, w, b), Y_lin, atol=1e-6), \
    "with alpha = 0 and data that IS linear, the fit should be exact: check the centring and b"
w, b = ridge_fit(emg[tr], pose[tr], alpha=10.0)
r_w, r_b = ref.ridge_fit(emg[tr], pose[tr], alpha=10.0)
assert np.allclose(w, r_w) and np.allclose(b, r_b), "weights differ from the reference"
sk = Ridge(alpha=10.0).fit(emg[tr], pose[tr])
assert np.allclose(ridge_predict(emg[te], w, b), sk.predict(emg[te])), "differs from scikit-learn's Ridge"
print("✅ Ridge works and matches scikit-learn")

### Task 3b · Choose α without peeking at the test set
`α` is a **hyperparameter**: the fit doesn't choose it, you do. If you tried several and kept the one with the
best **test** score, the test set would have helped train the model, and its score would be optimistic.

So split the **training** rows once more: within each recording, the first 80% of its training rows are
**inner-train** and the last 20% are **validation** (always later in time, like the real test set). Fit on
inner-train for each α, score on validation, keep the best α, then refit on all training rows with it.
`inner_split` (given below) makes that split for you; you write `choose_alpha`.

In [ ]:
def inner_split(rec_tr: np.ndarray, frac: float = 0.8) -> np.ndarray:
    """(T,) recording ids of the training rows → (T,) bool: True = inner-train, False = validation.
    Within each recording, the first `frac` of its rows are inner-train. (Given: nothing to fill in.)"""
    keep = np.zeros(len(rec_tr), dtype=bool)
    for r in np.unique(rec_tr):
        idx = np.flatnonzero(rec_tr == r)
        keep[idx[: int(frac * len(idx))]] = True
    return keep


ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)


def choose_alpha(F: np.ndarray, Y: np.ndarray, rec_tr: np.ndarray,
                 alphas: tuple[float, ...] = ALPHAS) -> float:
    """The alpha with the lowest validation angular error, using TRAINING rows only.

    Parameters
    ----------
    F      : (T, d) float, features of the training rows (e.g. emg[tr])
    Y      : (T, J) float, targets of the same rows (e.g. pose[tr])
    rec_tr : (T,) int, recording of each row (e.g. rec[tr])

    Returns
    -------
    float, one of `alphas`. Fit on inner_split(rec_tr) rows, score angular_error on the rest.
    """
    # TODO
    raise NotImplementedError


alpha_16 = None   # TODO: choose_alpha(emg[tr], pose[tr], rec[tr])
# TODO: results["16 ch, now"] = fit_and_score_ridge(emg[tr], pose[tr], emg[te], pose[te], alpha_16)

#### 💡 Stuck on Task 3b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `inner = inner_split(rec_tr)`: a True/False per row. `~inner` is the validation rows.
2. For each α: fit on `F[inner], Y[inner]`, predict `F[~inner]`, compute the angular error against `Y[~inner]`.
3. Return the α with the smallest error. Then use it for the 16-channel decoder on the real train/test split.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.argmin(errs)` → the position of the smallest value in a list.
- Unpacking a returned tuple into another call: `ridge_predict(F, *ridge_fit(F, Y, a))`.

```python
alphas = (0.1, 1.0, 10.0); errs = [3.2, 2.9, 3.5]
alphas[int(np.argmin(errs))]     # 1.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def choose_alpha(F, Y, rec_tr, alphas=ALPHAS):
    inner = inner_split(rec_tr)
    errs = []
    for a in alphas:
        w, b = ridge_fit(F[____], Y[____], a)
        errs.append(angular_error(Y[~inner], ridge_predict(F[____], w, b)))
    return float(alphas[int(np.argmin(____))])

alpha_16 = choose_alpha(emg[tr], pose[tr], rec[tr])
results["16 ch, now"] = fit_and_score_ridge(emg[tr], pose[tr], emg[te], pose[te], alpha_16)
```

</details>

In [ ]:
# ✅ Check Task 3b
assert alpha_16 == ref.choose_alpha(emg[tr], pose[tr], rec[tr]), \
    f"alpha_16 = {alpha_16}, reference {ref.choose_alpha(emg[tr], pose[tr], rec[tr])}"
assert np.isclose(results["16 ch, now"]["angle_deg"],
                  ref.fit_and_score_ridge(emg[tr], pose[tr], emg[te], pose[te], alpha_16)["angle_deg"])
train_fit = score(pose[tr], ridge_predict(emg[tr], *ridge_fit(emg[tr], pose[tr], alpha_16)))
for name in ["global median", "per-recording median", "16 ch, now"]:
    print(f"{name:22s} {results[name]['angle_deg']:6.2f}°   R² = {results[name]['r2']:6.3f}")
print(f"✅ alpha = {alpha_16}. On its OWN training rows the decoder gets R² = {train_fit['r2']:.3f}")

### 📊 What the instantaneous decoder predicts
True vs predicted angle of one joint over 30 s of test data. **What to look for:** does the prediction follow
the movements, or sit near the average pose and wiggle?

In [ ]:
j = 9                                         # MIDDLE_MCP_FE
w16, b16 = ridge_fit(emg[tr], pose[tr], alpha_16)
r_show = 0
sel = np.where(te & (rec == r_show))[0][:1500]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sel / 50, pose[sel, j], "k", lw=1, label="true")
ax.plot(sel / 50, ridge_predict(emg[sel], w16, b16)[:, j], "C1", lw=1, label="16 ch, now")
ax.set_xlabel("time (s)"); ax.set_ylabel("angle (°)"); ax.legend(); plt.tight_layout(); plt.show()

✍️ **Q3.** The decoder's R² on its **own training rows** is close to 0 too. Is it over-fitting or under-fitting? Using what
you know about muscles (EMG reflects force and *changes*; a held pose can need almost no activity), explain why a
linear map from the EMG **at this instant** to the joint angle **at this instant** can't work well.

> *Your answer:*

---
## Task 4 · Give the decoder the recent past of the EMG
Muscles fire *before* the finger moves, and a pose is the *result* of the recent history of activity. A linear
decoder can use that if we add **lagged copies** of the features as extra columns: `[X(t), X(t − L₁), X(t − L₂), …]`.

Two rules:
- Build the lags on the **full, time-ordered** arrays (all N frames), *then* select `tr` / `te` rows. Masking first
  would make "the previous row" a frame from somewhere else.
- A lag must **never reach into the previous recording**. Stop at the recording's first frame instead.

These lags only look into the **past**, so this decoder could run live (the causal-filtering idea from notebook 01).

**Log-EMG.** The envelope has rare huge bursts and long quiet stretches. `np.log(emg + 1e-3)` compresses the bursts
and stretches the quiet range, which suits a linear decoder better (muscle force grows less than linearly with
the envelope at high activity).

You'll do two things: a **lag sweep** (one lag at a time, 0 to 300 ms, to find *how far back* the information
is), then the **stacked** decoder with lags of 40, 100, 200 and 300 ms together.

In [ ]:
def lag_features(X: np.ndarray, rec: np.ndarray, lags: tuple[int, ...] = (5, 10, 20)) -> np.ndarray:
    """Stack the current features with lagged copies of themselves.

    Parameters
    ----------
    X    : (N, d) float, features for ALL frames, in time order
    rec  : (N,) int, recording of each frame
    lags : how many frames back each extra copy looks (50 frames = 1 s)

    Returns
    -------
    (N, d * (1 + len(lags))) float: [X(t), X(t - lags[0]), X(t - lags[1]), ...]
    where t - lag is clipped to the first frame of t's recording.
    """
    # TODO
    raise NotImplementedError


log_emg = np.log(emg + 1e-3)          # (N, 16)
LAG_SWEEP = [0, 2, 5, 10, 15]         # frames: 0, 40, 100, 200, 300 ms
LAGS = (2, 5, 10, 15)                 # the stacked decoder
sweep_err: list[float] = []           # test angular error for a decoder that sees ONLY log_emg(t - L)
# TODO: for each L in LAG_SWEEP: the features are log_emg shifted back by L frames
#       (lag_features(log_emg, rec, (L,))[:, 16:] for L > 0, log_emg itself for L = 0);
#       choose alpha on the tr rows, fit, append the test angular error
F_lag = None                          # (N, 16 * 5): TODO lag_features(log_emg, rec, LAGS)
alpha_lag = None                      # TODO: choose_alpha on F_lag[tr]
# TODO: results["16 ch + lags (log)"] = fit_and_score_ridge(F_lag[tr], pose[tr], F_lag[te], pose[te], alpha_lag)

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `lag_features`: work with frame positions `t = np.arange(N)`. The copy for lag `L` is `X[t - L]`, clipped to
   the recording's **first** frame: `X[np.maximum(t - L, first)]`. `np.hstack` everything side by side.
2. The sweep: for each `L`, build the shifted features, `choose_alpha`, `fit_and_score_ridge`, append `angle_deg`.
3. The stacked decoder: the same steps with `lag_features(log_emg, rec, LAGS)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- A recording starts where `rec` changes: `is_start = np.r_[True, rec[1:] != rec[:-1]]`.
- `np.maximum.accumulate(np.where(is_start, t, 0))` gives every frame the position of its recording's first frame.

```python
rec_toy = np.array([0, 0, 0, 1, 1]); t = np.arange(5)
is_start = np.r_[True, rec_toy[1:] != rec_toy[:-1]]        # [T, F, F, T, F]
np.maximum.accumulate(np.where(is_start, t, 0))            # [0, 0, 0, 3, 3]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def lag_features(X, rec, lags=(5, 10, 20)):
    t = np.arange(len(X))
    is_start = np.r_[True, rec[1:] != rec[:-1]]
    first = np.maximum.accumulate(np.where(is_start, ____, 0))
    return np.hstack([X] + [X[np.maximum(t - lag, ____)] for lag in ____])

for L in LAG_SWEEP:
    F_L = log_emg if L == 0 else lag_features(log_emg, rec, (L,))[:, ____:]
    a = choose_alpha(F_L[tr], pose[tr], rec[tr])
    sweep_err.append(fit_and_score_ridge(F_L[tr], pose[tr], F_L[te], pose[te], a)["____"])

F_lag = lag_features(log_emg, rec, ____)
alpha_lag = choose_alpha(F_lag[tr], pose[tr], rec[tr])
results["16 ch + lags (log)"] = fit_and_score_ridge(F_lag[____], pose[tr], F_lag[____], pose[te], alpha_lag)
```

</details>

In [ ]:
# ✅ Check Task 4
X_toy = np.arange(6.0)[:, None]; rec_toy = np.array([0, 0, 0, 1, 1, 1])
assert np.array_equal(lag_features(X_toy, rec_toy, (1, 2)), ref.lag_features(X_toy, rec_toy, (1, 2))), \
    f"got\n{lag_features(X_toy, rec_toy, (1, 2))}\nexpected\n{ref.lag_features(X_toy, rec_toy, (1, 2))}"
assert np.array_equal(F_lag, ref.lag_features(log_emg, rec, LAGS))
assert len(sweep_err) == len(LAG_SWEEP), "one error per lag in LAG_SWEEP"
F10 = ref.lag_features(log_emg, rec, (10,))[:, 16:]
a10 = ref.choose_alpha(F10[tr], pose[tr], rec[tr])
assert np.isclose(sweep_err[3], ref.fit_and_score_ridge(F10[tr], pose[tr], F10[te], pose[te], a10)["angle_deg"]), \
    "the 200 ms entry of the sweep differs from the reference"
for name in ["global median", "per-recording median", "16 ch, now", "16 ch + lags (log)"]:
    print(f"{name:22s} {results[name]['angle_deg']:6.2f}°   R² = {results[name]['r2']:6.3f}")
print("✅ Lag sweep (ms → °): " + ", ".join(f"{L * 20}: {e:.2f}" for L, e in zip(LAG_SWEEP, sweep_err)))

### 📊 How far back is the information?
Left: test error of a decoder that sees the EMG from **one** moment, L ms ago. Right: all decoders so far.
**What to look for:** where the sweep has its minimum. The *electromechanical delay* (EMG to force) is only
about 30–100 ms; a minimum much later than that means the pose depends on activity *built up* over time.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(np.array(LAG_SWEEP) * 20, sweep_err, marker="o")
ax[0].axhline(results["global median"]["angle_deg"], ls="--", color="gray", label="global median")
ax[0].axhline(results["per-recording median"]["angle_deg"], ls=":", color="gray", label="per-recording median")
ax[0].set_xlabel("lag L (ms)"); ax[0].set_ylabel("test angular error (°)"); ax[0].legend(fontsize=8)
names = list(results)
ax[1].barh(names, [results[n]["angle_deg"] for n in names], color="C0")
ax[1].set_xlabel("test angular error (°)"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()

✍️ **Q4.** Where is the minimum of the lag sweep, and how does it compare with the electromechanical delay (~30–100 ms)? The
stacked decoder beats every single lag. What does that say about *how* the pose depends on the EMG (one delayed
copy, or an accumulation over time)? Does it beat the **per-recording** median?

> *Your answer:*

---
## Task 5 · Angle or velocity: which does the EMG encode?
**H2** of the research plan predicts: *instantaneous EMG predicts joint **velocity** better than joint **angle**,
and EMG history helps angle more than velocity.* The reasoning: muscle activity makes force, force changes
movement, and the angle is the *sum* of past movement.

**Velocity from angles.** The central difference `v(t) = (pose(t+1) − pose(t−1)) / 2 × 50` (°/s) needs three
frames. It is only valid where frames t−1, t and t+1 are **all** valid and in the **same** recording: a difference
across a tracking dropout (pose = 0) or between two recordings would be a fake, huge velocity.

Compare with **R²** (angular error is in different units for angles and velocities), on the frames where
**both** targets are valid, so the comparison is like for like.

In [ ]:
def joint_velocity(pose: np.ndarray, rec: np.ndarray, valid: np.ndarray, fs: int = 50
                   ) -> tuple[np.ndarray, np.ndarray]:
    """Central-difference joint velocity, never across a gap or a recording boundary.

    Parameters
    ----------
    pose  : (N, J) float, angles in degrees, time-ordered
    rec   : (N,) int, recording of each frame
    valid : (N,) bool, pose_valid

    Returns
    -------
    V  : (N, J) float, (pose(t+1) - pose(t-1)) / 2 * fs in °/s; 0 where not ok
    ok : (N,) bool, True only where t-1, t, t+1 are all valid AND in the same recording (False at both ends)
    """
    # TODO
    raise NotImplementedError


V, v_ok = None, None                        # TODO: joint_velocity(pose, rec, pose_valid)
trv, tev = tr & v_ok if v_ok is not None else None, te & v_ok if v_ok is not None else None
r2_table: dict[tuple[str, str], float] = {}  # (features, target) → test R², e.g. ("now", "velocity")
# TODO: for features in {"now": log_emg, "lags": F_lag} and target in {"angle": pose, "velocity": V}:
#       choose alpha on the trv rows, fit on trv, r2_table[(features, target)] = r2 on the tev rows

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `joint_velocity`: `ok[1:-1]` combines four conditions with `&`: frames `2:`, `1:-1` and `:-2` valid, and the
   three recordings equal. `V[1:-1] = (pose[2:] - pose[:-2]) / 2 * fs`; then zero the rows that aren't ok.
2. Recompute `trv` / `tev` after you fill `V, v_ok` (the line below the TODO does it once `v_ok` exists; rerun the cell).
3. Four fits: a double loop over the two feature sets and the two targets.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Shifted slices line up neighbours: `a[2:]` is "next", `a[:-2]` is "previous", `a[1:-1]` is "now".
- `V[~ok] = 0.0` zeroes every row where `ok` is False.

```python
p = np.array([0.0, 1.0, 4.0, 9.0])
(p[2:] - p[:-2]) / 2          # array([2., 4.]): central differences at positions 1 and 2
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def joint_velocity(pose, rec, valid, fs=50):
    V = np.zeros_like(pose); ok = np.zeros(len(pose), dtype=bool)
    same = (rec[2:] == rec[1:-1]) & (rec[1:-1] == rec[____])
    ok[1:-1] = valid[2:] & valid[1:-1] & valid[____] & same
    V[1:-1] = (pose[____] - pose[:-2]) / 2 * fs
    V[~ok] = 0.0
    return V, ok

V, v_ok = joint_velocity(pose, rec, pose_valid)
trv, tev = tr & v_ok, te & v_ok
for fname, F in {"now": log_emg, "lags": F_lag}.items():
    for tname, Y in {"angle": pose, "velocity": V}.items():
        a = choose_alpha(F[trv], Y[trv], rec[trv])
        w, b = ridge_fit(F[trv], Y[trv], a)
        r2_table[(fname, tname)] = r2(Y[tev], ridge_predict(F[____], w, b))
```

</details>

In [ ]:
# ✅ Check Task 5
p_toy = np.array([[0.0], [1.0], [4.0], [9.0], [16.0], [0.0], [1.0]])
rec_v = np.array([0, 0, 0, 0, 1, 1, 1]); val_v = np.array([True, True, True, True, True, True, True])
Vt, okt = joint_velocity(p_toy, rec_v, val_v, fs=1)
rVt, rokt = ref.joint_velocity(p_toy, rec_v, val_v, fs=1)
assert np.array_equal(okt, rokt), f"ok = {okt}, expected {rokt}: never across a recording boundary"
assert np.allclose(Vt, rVt), f"V = {Vt.ravel()}, expected {rVt.ravel()}"
rV, rok = ref.joint_velocity(pose, rec, pose_valid)
assert np.array_equal(v_ok, rok) and np.allclose(V, rV)
assert set(r2_table) == {("now", "angle"), ("now", "velocity"), ("lags", "angle"), ("lags", "velocity")}
print("✅ Test R² (same frames for both targets):")
for (f, t), v in r2_table.items():
    print(f"   {f:5s} → {t:9s} {v:6.3f}")

### 📊 Angle vs velocity
**What to look for:** H2 predicted the *now* bars to be higher for velocity than for angle, and lags to help angle
more. Does the data agree?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
x = np.arange(2)
for i, t in enumerate(["angle", "velocity"]):
    ax.bar(x + (i - 0.5) * 0.4, [r2_table[("now", t)], r2_table[("lags", t)]], 0.4, label=t)
ax.set_xticks(x, ["EMG now", "EMG + lags"]); ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("test R²"); ax.legend(); plt.tight_layout(); plt.show()

✍️ **Q5.** Did the data support H2 (velocity easier than angle from the instantaneous EMG)? If velocity is *harder*, give two
possible reasons. (Hint: the pose is now low-passed at 6 Hz, but what does a central difference still do to
whatever tracking noise is left? And what does a held, *static* pose look like in velocity?)

> *Your answer:*

---
## Task 6 · The honest test: blocks, a shifted null, and held-out stages
One number on one split can mislead. This task asks three questions, each answered **block by block** with a
95% confidence interval over blocks:

1. **Time split, paired:** for each block, decoder error minus baseline error. Is the mean difference below 0,
   with a CI that excludes 0? Against the **global** median *and* against the **per-recording** median.
2. **Shifted null:** train and test the same decoder on `protocol.circular_shift_null` EMG. If the null decoder
   does as well as the real one, the "decoding" came from something other than EMG–pose alignment.
3. **Held-out stages:** `protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename)` gives
   **14 folds**. (The two free-style blocks are one fold: in each, one hand did "OneHandedFreeStyle" while the
   idle hand's recording is labelled "unconstrained", and simultaneous recordings must never be split between
   train and test.) In each fold, fit on the other stages and score the held-out one. The baseline here is the
   **global** median of the training stages: a new task has no training frames of its own.

Keep `alpha_lag` fixed (chosen on the time split's training rows), so the folds only refit the weights.

In [ ]:
def loso_errors(F: np.ndarray, Y: np.ndarray, alpha: float) -> dict[str, tuple[float, float]]:
    """Leave-one-stage-out: decoder vs global-median baseline on each held-out stage group.

    Parameters
    ----------
    F     : (N, d) float, features for all frames
    Y     : (N, 20) float, pose
    alpha : ridge penalty (kept fixed)

    Returns
    -------
    {fold name: (decoder angular error, baseline angular error)} for the 14 folds of
    protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename).
    Decoder = ridge fit on the train frames; baseline = median of Y over the train frames.
    """
    # TODO
    raise NotImplementedError


# 1. time split, block by block
e_lag = None    # (15,): TODO per_block_error(pose, prediction of the lagged decoder (all N rows), te, block)
# 2. shifted null: the same decoder on circularly shifted EMG
emg_null = protocol.circular_shift_null(emg, rec, np.random.default_rng(SEED))
F_null = None   # TODO: lag_features(np.log(emg_null + 1e-3), rec, LAGS)
e_null = None   # (15,): TODO per_block_error for the null decoder (fit on F_null[tr], alpha_lag)
# 3. held-out stages
loso_real = None   # TODO: loso_errors(F_lag, pose, alpha_lag)
loso_null = None   # TODO: loso_errors(F_null, pose, alpha_lag)

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `loso_errors`: loop over the folds; `name, trm, tem` each time; skip a fold with no test frames; fit ridge on
   `F[trm], Y[trm]`; the decoder error is on `tem`; the baseline is `np.median(Y[trm], axis=0)` scored on `tem`.
2. Time split: predict **all** N rows with the lagged decoder (`ridge_predict(F_lag, w, b)`), then `per_block_error`.
3. Null: the same three lines with `F_null` in place of `F_lag`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `for name, trm, tem in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename):`
- A baseline prediction for the test frames: `np.tile(np.median(Y[trm], axis=0), (tem.sum(), 1))`.

```python
d_toy = {"a": (1.0, 2.0), "b": (3.0, 3.5)}
np.array([v[0] - v[1] for v in d_toy.values()])     # array([-1. , -0.5]): decoder minus baseline
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def loso_errors(F, Y, alpha):
    out = {}
    for name, trm, tem in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename):
        if tem.sum() == 0:
            continue
        w, b = ridge_fit(F[____], Y[____], alpha)
        dec = angular_error(Y[tem], ridge_predict(F[____], w, b))
        base = angular_error(Y[tem], np.tile(np.median(Y[____], axis=0), (tem.sum(), 1)))
        out[name] = (dec, base)
    return out

w_l, b_l = ridge_fit(F_lag[tr], pose[tr], alpha_lag)
e_lag = per_block_error(pose, ridge_predict(____, w_l, b_l), te, block)
F_null = lag_features(np.log(emg_null + 1e-3), rec, LAGS)
w_n, b_n = ridge_fit(F_null[tr], pose[tr], alpha_lag)
e_null = per_block_error(pose, ridge_predict(____, w_n, b_n), te, block)
loso_real = loso_errors(F_lag, pose, alpha_lag)
loso_null = loso_errors(____, pose, alpha_lag)
```

</details>

In [ ]:
# ✅ Check Task 6
assert e_lag.shape == (N_BLOCKS,) and e_null.shape == (N_BLOCKS,)
assert np.allclose(F_null, ref.lag_features(np.log(emg_null + 1e-3), rec, LAGS))
folds = list(protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename))
assert len(loso_real) == sum(te_.sum() > 0 for _, _, te_ in folds), "one entry per fold that has test frames"
name0, trm0, tem0 = next(f for f in folds if f[2].sum() > 0)
w0, b0 = ref.ridge_fit(F_lag[trm0], pose[trm0], alpha_lag)
assert np.isclose(loso_real[name0][0], ref.angular_error(pose[tem0], ref.ridge_predict(F_lag[tem0], w0, b0)))
assert np.isclose(loso_real[name0][1], ref.angular_error(pose[tem0], np.tile(np.median(pose[trm0], axis=0), (tem0.sum(), 1))))

def show(label, diff):
    m, lo, hi = protocol.session_bootstrap_ci(diff)
    verdict = "better" if hi < 0 else ("worse" if lo > 0 else "no clear difference")
    print(f"{label:46s} {m:+6.2f}°  (95% CI {lo:+6.2f} to {hi:+6.2f})  → {verdict}")
    return m, lo, hi

real = np.array([v[0] for v in loso_real.values()]); base_l = np.array([v[1] for v in loso_real.values()])
null = np.array([loso_null[k][0] for k in loso_real])
stage4 = {
    "time split: lags − global median": show("time split: lags − global median (per block)", e_lag - e_glob),
    "time split: lags − per-recording median": show("time split: lags − per-recording median", e_lag - e_rec),
    "time split: lags − shifted null": show("time split: lags − shifted null", e_lag - e_null),
    "held-out stages: lags − global median": show("held-out stages: lags − global median (per fold)", real - base_l),
    "held-out stages: lags − shifted null": show("held-out stages: lags − shifted null", real - null),
}
print("✅ Stage 4 numbers ready for the gate table")

### 📊 Paired differences with confidence intervals
Each row is one comparison: the dot is the mean difference (decoder minus reference, in degrees), the bar its 95%
CI over blocks or folds. **What to look for:** a bar entirely **left of 0** means the decoder is reliably better.

In [ ]:
labels = list(stage4)
fig, ax = plt.subplots(figsize=(8, 3.2))
for i, k in enumerate(labels):
    m, lo, hi = stage4[k]
    ax.plot([lo, hi], [i, i], color="C0", lw=2); ax.plot(m, i, "o", color="C0")
ax.axvline(0, color="k", lw=0.8); ax.set_yticks(range(len(labels)), labels, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("decoder minus reference (°): left of 0 = decoder better"); plt.tight_layout(); plt.show()

✍️ **Q6.** On held-out stages, which wins: the lagged decoder, the global median, or the shifted null? On the time split,
does the decoder beat the **per-recording** median? Put the two together: is the decoder tracking the hand, or
mostly recognising the task?

> *Your answer:*

---
## Task 7 · Encoding vs decoding
**Decoding** asks: from the EMG, what is the pose? **Encoding** asks the reverse: from the pose, what is the EMG?

The research plan predicts encoding is *worse*, because one pose can be held with very different muscle activity
(relaxed, or with antagonist muscles co-contracting to stiffen the hand). The pose then doesn't determine the EMG,
while the EMG still says something about the pose.

Fit ridge from the pose (now, and with the same lags) to `log_emg`, on the time split, and compare its R² with the
decoding R² from Task 5.

In [ ]:
enc_r2: dict[str, float] = {}     # "now" / "lags" → test R² of predicting log_emg from the pose
# TODO: features pose (now) and lag_features(pose, rec, LAGS) (lags); target log_emg;
#       choose alpha on tr, fit on tr, r2 on te; store in enc_r2

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Two feature sets: `pose` and `lag_features(pose, rec, LAGS)`.
2. For each: `choose_alpha` (targets = `log_emg[tr]`), `ridge_fit`, `r2` on the test rows.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Nothing new: Task 3b with the roles of EMG and pose swapped.

```python
for name, F in {"now": pose}.items():
    print(name, F.shape)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for name, Fp in {"now": pose, "lags": lag_features(pose, rec, ____)}.items():
    a = choose_alpha(Fp[tr], log_emg[tr], rec[tr])
    w, b = ridge_fit(Fp[tr], ____[tr], a)
    enc_r2[name] = r2(log_emg[te], ridge_predict(Fp[____], w, b))
```

</details>

In [ ]:
# ✅ Check Task 7
Fp_ref = ref.lag_features(pose, rec, LAGS)
a_ref = ref.choose_alpha(Fp_ref[tr], log_emg[tr], rec[tr])
w_ref, b_ref = ref.ridge_fit(Fp_ref[tr], log_emg[tr], a_ref)
assert np.isclose(enc_r2["lags"], ref.r2(log_emg[te], ref.ridge_predict(Fp_ref[te], w_ref, b_ref)))
print(f"✅ Encoding R²: now {enc_r2['now']:.3f}, lags {enc_r2['lags']:.3f}   |   "
      f"decoding R² (angle): now {r2_table[('now', 'angle')]:.3f}, lags {r2_table[('lags', 'angle')]:.3f}")

✍️ **Q7.** Is encoding (pose → EMG) better or worse than decoding (EMG → pose)? What does that say about the co-contraction
argument? Name one thing that would make the EMG of a held pose vary that the pose can't see.

> *Your answer:*

---
## Result · gate for stage 4
**Question:** Does the EMG (with its recent history) carry pose information beyond the baselines and the shifted null?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Lagged decoder − global median (time split) | | | |
| Lagged decoder − per-recording median (time split) | | | |
| Lagged decoder − shifted null (time split) | | | |
| Lagged decoder − global median (held-out stages) | | | |
| Lagged decoder − shifted null (held-out stages) | | | |
| Best single lag (ms) | | | |
| R²: angle vs velocity (EMG now / with lags) | | | |
| Encoding vs decoding R² | | | |

**Gate:** the lagged decoder beats the global median **and** the shifted null on held-out stages (CIs below 0). Record separately whether it beats the per-recording median on the time split.

- **Pass →** stage 4b (notebook 06): find *which part* of the EMG carries the pose (the shared EMG–pose subspace); then stage 5 (notebook 07) models the dynamics, and stage 6 (notebook 08) tests the bottlenecks with this decoder.
- **Fail →** the EMG envelope, as preprocessed, carries no usable pose information for a linear decoder: revisit the preprocessing (notebook 01) or move to nonlinear / dynamical decoders before any bottleneck study.
- **Was your prediction right? Why or why not?**

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person, 15 blocks** | Every CI is over 15 blocks (14 stage folds); the two hands of a block are simultaneous, so they are one unit | Repeat on several users of the full dataset |
| **Time split vs held-out stages** | The time split tests the same task later; held-out stages test a new task. Results can differ a lot | Always report both; trust held-out stages for generalisation |
| **Per-recording median is strong** | Much of the pose variance is *which task*; a decoder can look good by recognising the task | Compare against the per-recording median on the time split |
| **Zero-phase envelope** | The 10 Hz smoothing in notebook 01 uses a little *future* EMG (~100 ms), so 'past-only' lags are not perfectly causal | Rebuild the envelope with a causal filter and rerun Tasks 4–6 |
| **Pose cleaning** | The pose is low-passed at 6 Hz and dropouts are padded (data version 2). Velocity still amplifies any residual tracking noise, and the cut-off removes the fastest flicks | Repeat Task 5 with a 10 Hz cut-off in prep.py |
| **Alpha chosen once** | alpha_lag comes from the time split and is reused in the folds; a per-fold choice could change errors slightly | Nested selection inside each fold |
| **Linear decoder** | Ridge can only add up features; the true EMG→pose relation is dynamical and nonlinear | Stage 4b (shared subspace, notebook 06) and stage 5 (dynamics, notebook 07) |
| **One shifted null** | A single random shift per recording can be lucky or unlucky | Repeat the null with several seeds and average |